In [2]:
from pathlib import Path
import pandas as pd
import tkinter as tk
from tkinter import filedialog

from pathlib import Path
import tkinter as tk
from tkinter import filedialog

# -----------------------------
# 1. Select multiple trajectory CSV files from different folders
# -----------------------------

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

selected_files = []

while True:
    files = filedialog.askopenfilenames(
        title="Select trajectory CSV files, then click Cancel when finished",
        filetypes=[
            ("CSV files", "*.csv"),
            ("All files", "*.*")
        ]
    )

    # If user clicks Cancel, stop selecting
    if not files:
        break

    selected_files.extend(files)

root.destroy()

# Remove duplicate files if accidentally selected twice
selected_files = list(dict.fromkeys(selected_files))

if not selected_files:
    raise FileNotFoundError("No CSV files selected.")

csv_paths = [Path(f) for f in selected_files]

print(f"Selected {len(csv_paths)} CSV files:")
for p in csv_paths:
    print(" -", p)



Selected 18 CSV files:
 - D:\Dharaneeswar\data\1-1 Masks\44-60\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\172-181\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\215-232\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\279-296\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\344-360\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\401-418\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\446-463\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\671-687\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\718-736\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\760-778\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\1126-1145\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\1168-1187\trajectory.csv
 - D:\Dharaneeswar\data\1-1 Masks\1232-1249\trajectory.csv
 - D:\Dharaneeswar\data\2-1 Masks\30-45\trajectory_30-45.csv
 - D:\Dharaneeswar\data\3-1 Masks\19-34\trajectory_19.csv
 - D:\Dharaneeswar\data\3-2 Masks\15-23\trajectory-15-22.csv
 - D:\Dharaneeswar\data\3-2 Masks\77-86\trajectory_7

In [5]:
# -----------------------------
# 2. Extract point with smallest Longitudinal_From_Special
#    and format using same heading as trapping_points.csv
# -----------------------------

longitudinal_col = "Longitudinal_From_Special"

min_points = []

for csv_path in csv_paths:
    df = pd.read_csv(csv_path)

    if longitudinal_col not in df.columns:
        print(f"Skipped {csv_path.name}: missing column '{longitudinal_col}'")
        continue

    valid_df = df.dropna(subset=[longitudinal_col]).copy()

    if valid_df.empty:
        print(f"Skipped {csv_path.name}: no valid '{longitudinal_col}' values")
        continue

    # Find row with minimum Longitudinal_From_Special
    min_idx = valid_df[longitudinal_col].idxmin()
    min_row = valid_df.loc[min_idx]

    # Save using same heading as trapping_points.csv
    out_row = {
        "source_csv": str(csv_path),
        "row_index": min_idx,
        "Trajectory_ID": min_row.get("Trajectory_ID", ""),
        "Frame": min_row.get("Frame", ""),
        "Longitudinal_From_Special": min_row.get("Longitudinal_From_Special", ""),
        "Transverse_Dist": min_row.get("Transverse_Dist", ""),
        "label": "",
        "source_file": csv_path.name,
    }

    min_points.append(out_row)

if len(min_points) == 0:
    raise ValueError("No valid minimum points found from selected CSV files.")

min_points_df = pd.DataFrame(min_points)

# Force exact column order to match trapping_points.csv
output_columns = [
    "source_csv",
    "row_index",
    "Trajectory_ID",
    "Frame",
    "Longitudinal_From_Special",
    "Transverse_Dist",
    "label",
    "source_file",
]

min_points_df = min_points_df[output_columns]

print(f"Extracted {len(min_points_df)} minimum-longitudinal points.")
display(min_points_df)



Extracted 18 minimum-longitudinal points.


,source_csv,row_index,Trajectory_ID,Frame,Longitudinal_From_Special,Transverse_Dist,label,source_file
0,D:\Dharaneeswar\data\1-1 Masks\44-60\trajector...,12,,56.0,28.980493,-2.233705,,trajectory.csv
1,D:\Dharaneeswar\data\1-1 Masks\172-181\traject...,7,,179.0,145.525680,-0.451647,,trajectory.csv
2,D:\Dharaneeswar\data\1-1 Masks\215-232\traject...,15,,230.0,-10.190847,18.474500,,trajectory.csv
3,D:\Dharaneeswar\data\1-1 Masks\279-296\traject...,14,,293.0,-3.958172,-19.866219,,trajectory.csv
4,D:\Dharaneeswar\data\1-1 Masks\344-360\traject...,12,,356.0,-19.296647,6.617308,,trajectory.csv
5,D:\Dharaneeswar\data\1-1 Masks\401-418\traject...,10,,411.0,-13.456071,13.631283,,trajectory.csv
6,D:\Dharaneeswar\data\1-1 Masks\446-463\traject...,12,,458.0,-16.066902,1.406899,,trajectory.csv
7,D:\Dharaneeswar\data\1-1 Masks\671-687\traject...,12,,683.0,-9.167181,-8.092571,,trajectory.csv
8,D:\Dharaneeswar\data\1-1 Masks\718-736\traject...,15,,733.0,6.732814,-19.302975,,trajectory.csv
9,D:\Dharaneeswar\data\1-1 Masks\760-778\traject...,10,,770.0,-39.177053,-6.692494,,trajectory.csv


In [6]:
# -----------------------------
# 3. Export result
# -----------------------------

output_path = Path("min_longitudinal_points_from_multiple_csvs.csv")
min_points_df.to_csv(output_path, index=False)

print(f"Exported result to: {output_path.resolve()}")

Exported result to: D:\Dharaneeswar\sam2GUI_DD\min_longitudinal_points_from_multiple_csvs.csv
